# PINN Training & Lambda Ablation — Low-Dose CT (Colab GPU)

Week 3 of the proposal: implement the differentiable sinogram-consistency loss,
sweep the physics weight lambda to find a stable regime, then train the winner
properly and compare it against the Week 2 baseline.

**Before running:** Runtime -> Change runtime type -> **GPU** (T4 is fine).

Run `train_baseline.ipynb` first — this notebook compares against `models/baseline.pt`.

## 1. Get the code and confirm the GPU

In [ ]:
import os

REPO = 'PINN-CT-Denoising-Web-App'
PATH = f'/content/{REPO}'

# Idempotent: clone on a fresh runtime, otherwise just pull the latest commits.
if not os.path.exists(PATH):
    !git clone https://github.com/KendoCee25/{REPO}.git {PATH}

%cd {PATH}
!git pull
!git log --oneline -1

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!pip install -q scikit-image

## 2. Regenerate the dataset **with sinograms**

The physics loss needs the measured projection data, and it cannot be recovered
from the noisy image after the fact (FBP followed by Radon band-limits, it is not
the identity). The Week 2 dataset did not store it, so regenerate.

Same seed and parameters as Week 2, and the RNG is consumed in the same order, so
the clean/noisy images come out identical — **the existing `baseline.pt` stays a
valid comparison** and does not need retraining. Only the `sino` array is new.

In [ ]:
!python -m training.make_dataset --n-phantoms 500 --size 128 --device cuda --out data/dataset

import json
m = json.load(open('data/dataset/manifest.json'))
print()
print('manifest:', m)
assert m['has_sinograms'], 'dataset has no sinograms — the PINN cannot be trained'

## 3. Lambda ablation

Sweeps lambda in {0, 0.01, 0.1, 0.5, 1.0} under identical conditions and reports
which values train stably. Short runs — the goal is to locate a stable regime
cheaply, then retrain the winner properly in step 4.

**lambda = 0 is the control.** It runs the same code path with the physics term
switched off, so the comparison isolates the loss from everything else that
changed since Week 2 (gradient clipping, the batch refactor). That is a fairer
reference than the Week 2 checkpoint.

The physics term uses 60 of 180 angles — the proposal's stated mitigation for a
slow forward projection. Raise `--loss-angles` to 180 if time allows.

In [ ]:
!python -m training.ablate --data-dir data/dataset --epochs 15 --batch-size 32 \
    --loss-angles 60 --device cuda --out-dir models

### Plot the ablation

In [ ]:
import json
import matplotlib.pyplot as plt

abl = json.load(open('models/ablation.json'))
ok = [r for r in abl if not r['diverged']]
lams = [r['lam'] for r in ok]
psnr_vals = [r['best_val_psnr'] for r in ok]
deltas = [r['per_dose']['all']['delta_psnr'] for r in ok]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(lams, psnr_vals, 'o-')
ax[0].set_xlabel('lambda'); ax[0].set_ylabel('best val PSNR (dB)')
ax[0].set_title('Validation PSNR vs physics weight'); ax[0].set_xscale('symlog', linthresh=0.01)
ax[1].plot(lams, deltas, 'o-', color='tab:green')
ax[1].axhline(0, color='r', ls='--', lw=1)
ax[1].set_xlabel('lambda'); ax[1].set_ylabel('delta PSNR vs noisy input (dB)')
ax[1].set_title('Improvement over doing nothing'); ax[1].set_xscale('symlog', linthresh=0.01)
plt.tight_layout(); plt.show()

diverged = [r['lam'] for r in abl if r['diverged']]
print('diverged lambdas:', diverged if diverged else 'none')

## 4. Train the final PINN at the best lambda

Full-length run, matching the baseline's 40 epochs so the comparison is like for like.

In [ ]:
import json

abl = json.load(open('models/ablation.json'))
stable = [r for r in abl if not r['diverged'] and r['lam'] > 0]
best_lam = max(stable, key=lambda r: r['best_val_psnr'])['lam']
print('Training final PINN at lambda =', best_lam)

!python -m training.train --data-dir data/dataset --epochs 40 --batch-size 32 \
    --lr 1e-3 --lam {best_lam} --loss-angles 60 --device cuda --out models/pinn.pt

## 5. Baseline vs PINN — the headline comparison

Both judged against the untouched noisy input. This table is the core result of
the dissertation and drives the web app's "winner" badge.

In [ ]:
import numpy as np
import torch
from training.unet import UNet
from training.metrics import psnr, ssim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
test = np.load('data/dataset/test.npz')
clean, noisy, dose = test['clean'], test['noisy'], test['dose']
DOSE_NAMES = {0: 'low', 1: 'medium', 2: 'high'}

def load(path):
    ck = torch.load(path, map_location=device)
    m = UNet().to(device); m.load_state_dict(ck['model_state']); m.eval()
    return m

models = {'baseline': load('models/baseline.pt'), 'PINN': load('models/pinn.pt')}

@torch.no_grad()
def run(model, idx):
    outs = []
    for s in range(0, len(idx), 64):
        b = idx[s:s + 64]
        x = torch.from_numpy(noisy[b])[:, None].to(device)
        outs.append(model(x).clamp(0, 1).cpu().numpy()[:, 0])
    return np.concatenate(outs)

print(f"{'dose':<9}{'input':<13}{'baseline':<19}{'PINN':<19}{'PINN - baseline'}")
print('-' * 74)
for code in [0, 1, 2, None]:
    idx = np.arange(len(clean)) if code is None else np.where(dose == code)[0]
    in_p = np.mean([psnr(noisy[i], clean[i]) for i in idx])
    row = {}
    for name, model in models.items():
        y = run(model, idx)
        p = np.mean([psnr(y[k], clean[i]) for k, i in enumerate(idx)])
        s = np.mean([ssim(y[k], clean[i]) for k, i in enumerate(idx)])
        row[name] = (p, s)
    gap = row['PINN'][0] - row['baseline'][0]
    name = 'ALL' if code is None else DOSE_NAMES[code]
    print(f"{name:<9}{in_p:6.2f} dB    "
          f"{row['baseline'][0]:6.2f} / {row['baseline'][1]:.4f}    "
          f"{row['PINN'][0]:6.2f} / {row['PINN'][1]:.4f}    {gap:+.2f} dB")

### Qualitative: noisy / baseline / PINN / clean

In [ ]:
import matplotlib.pyplot as plt

low_idx = np.where(dose == 0)[0]  # low dose is where the physics term should help most
fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for row in range(3):
    i = int(np.random.choice(low_idx))
    x = torch.from_numpy(noisy[i])[None, None].to(device)
    with torch.no_grad():
        b_out = models['baseline'](x).clamp(0, 1).cpu().numpy()[0, 0]
        p_out = models['PINN'](x).clamp(0, 1).cpu().numpy()[0, 0]
    panels = [(noisy[i], 'noisy'), (b_out, 'baseline'), (p_out, 'PINN'), (clean[i], 'clean')]
    for ax, (img, title) in zip(axes[row], panels):
        ax.imshow(img, cmap='gray', vmin=0, vmax=1); ax.axis('off')
        if title == 'clean':
            ax.set_title('clean (ground truth)')
        else:
            ax.set_title(f'{title}\nPSNR {psnr(img, clean[i]):.1f} | SSIM {ssim(img, clean[i]):.3f}')
plt.tight_layout(); plt.show()

## 6. Download
Save into the repo's `models/` folder for the backend (Weeks 6-7).

In [ ]:
from google.colab import files
files.download('models/pinn.pt')
files.download('models/pinn.curves.json')
files.download('models/ablation.json')